# External six-page diagnostic (AI labels only)

All three fine-tuned MPNet models and matched frozen controls, unchanged weights and validation thresholds. Six AI-reviewed pages form three dependent original/injected pairs. This does not satisfy the pending human-review or unseen-family checks. No retraining, threshold selection or population performance claim.

GPU: user-requested A100. Inputs and runner are frozen in `diagnostic/protocol.json`. Setup pins the same package versions as the preceding experiment; see the retained setup log.

Upload the frozen small input archive as `/content/external_diagnostic_inputs.tar.gz` and each unchanged fine-tuned checkpoint as 53 numbered 8-MiB parts (`external_weight_SEED_000.part` onward). The last part is shorter. Reassembly requires each frozen SHA256 before scoring.


In [ ]:
import os, subprocess, sys
os.environ.update(USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false')
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchvision','torchaudio'],check=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','torch==2.9.1','transformers==4.57.3','sentence-transformers==5.1.2','numpy==2.3.5','scikit-learn==1.7.2','beautifulsoup4==4.14.2','scipy==1.16.3'],check=True)
# Use fresh processes after package installation; the notebook may retain old NumPy modules.
check="import torch; assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(); print('Verified GPU',torch.cuda.get_device_name())"
print(subprocess.check_output([sys.executable,'-c',check],text=True))


In [ ]:
from pathlib import Path
import hashlib, json, shutil, tarfile
root=Path('/content/external_diagnostic')
root.mkdir(exist_ok=True)
with tarfile.open('/content/external_diagnostic_inputs.tar.gz') as archive:
    archive.extractall(root, filter='data')
models=root/'results/supervised_schedule_20260911/completed/mpnet_supervised_schedule_replay'
manifest=json.loads((root/'results/generalization_20260911/models_frozen.json').read_text())
for seed in (17,42,73):
    parts=sorted(Path('/content').glob(f'external_weight_{seed}_*.part'))
    assert len(parts)==53
    target=models/f'finetuned_{seed}/checkpoint.pt'
    with target.open('wb') as output:
        for part in parts:
            with part.open('rb') as source: shutil.copyfileobj(source,output)
    with target.open('rb') as stream: actual=hashlib.file_digest(stream,'sha256').hexdigest()
    assert actual==manifest['models'][f'finetuned_{seed}']['checkpoint_sha256']
    print('Restored unchanged checkpoint',seed,actual,flush=True)


In [ ]:
import os, subprocess
os.environ.update(USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false')
subprocess.run(['python','-u','scripts/run_external_diagnostic.py','run'],cwd=root,check=True)
print('EXTERNAL DIAGNOSTIC COMPLETE',flush=True)
